### 1：Imports & Configuration

In [25]:
import pandas as pd
import xml.etree.ElementTree as ET
import xml.etree.ElementTree as ET
import zipfile
import sqlite3
import os
from lxml import etree as LET
from tqdm import tqdm

ZIP_PATH = "../data/UDID_FULL_RELEASE_20260501.zip"
DB_PATH  = "../database/chinese_medical_devices.db"

column_mapping = {
    'zxxsdycpbs': 'udi_code',
    'cpbsbmtxmc': 'coding_system',
    'cpbsfbrq': 'udi_publish_date',
    'zxxsdyzsydydsl': 'units_per_package',
    'sydycpbs': 'unit_of_use_code',
    'bszt': 'barcode_type',
    'sfyzcbayz': 'consistent_with_registration',
    'zcbacpbs': 'registration_product_code',
    'sfybtzjbs': 'has_companion_device_code',
    'btcpbsyzxxsdycpbssfyz': 'companion_code_verified',
    'btcpbs': 'companion_device_code',
    'cpmctymc': 'product_generic_name',
    'spmc': 'trade_name',
    'ggxh': 'model_specification',
    'sfwblztlcp': 'is_kit_or_set',
    'cpms': 'product_description',
    'cphhhbh': 'product_catalog_number',
    'yflbm': 'old_classification_code',
    'qxlb': 'device_category',
    'flbm': 'classification_code',
    'tyshxydm': 'unified_social_credit_code',
    'zczbhhzbapzbh': 'registration_or_filing_number',
    'ylqxzcrbarmc': 'registrant_name_cn',
    'ylqxzcrbarywmc': 'registrant_name_en',
    'ybbm': 'medical_insurance_code',
    'cplb': 'product_type',
    'cgzmraqxgxx': 'mr_safety_info',
    'sfbjwycxsy': 'is_single_use',
    'zdcfsycs': 'max_reuse_times',
    'sfwwjbz': 'is_sterile_package',
    'syqsfxyjxmj': 'requires_sterilization_before_use',
    'mjfs': 'sterilization_method',
    'qtxxdwzlj': 'additional_info_url',
    'tsrq': 'special_date',
    'scbssfbhph': 'label_includes_lot_number',
    'scbssfbhxlh': 'label_includes_serial_number',
    'scbssfbhscrq': 'label_includes_manufacture_date',
    'scbssfbhsxrq': 'label_includes_expiry_date',
    'tscchcztj': 'special_storage_conditions',
    'tsccsm': 'special_storage_notes',
    'deviceRecordKey': 'device_record_key',
    'versionNumber': 'version_number',
    'versionTime': 'version_date',
    'versionStauts': 'version_status',
    'correctionNumber': 'correction_count',
    'correctionRemark': 'correction_remark',
    'correctionTime': 'correction_date',
}

date_columns = ['udi_publish_date', 'version_date', 'correction_date', 'special_date']

# Deduplication key: same license number + same device + same manufacturer
duplicate_keys = ['device_record_key']

### 2：Parse Function

In [4]:
def parse_xml_to_df(root):
    records = []
    for device in root.find('devices'):
        record = {}
        for child in device:
            if child.tag == 'contactList':
                contact = child.find('.//contact')
                if contact is not None:
                    record['contact_fax']   = contact.findtext('qylxrcz')
                    record['contact_email'] = contact.findtext('qylxryx')
                    record['contact_phone'] = contact.findtext('qylxrdh')
                else:
                    record['contact_fax'] = record['contact_email'] = record['contact_phone'] = None
            else:
                record[child.tag] = child.text if child.text and child.text.strip() else None
        records.append(record)
    return pd.DataFrame(records)

### 3： Stream All XMLs into SQLite

In [6]:
# Drop and recreate table for a clean run
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA journal_mode=WAL")
conn.execute("PRAGMA synchronous=NORMAL")
conn.execute("DROP TABLE IF EXISTS devices")
conn.commit()

total_written = 0
error_files   = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    xml_names = sorted([n for n in zf.namelist() if n.endswith('.xml')])

    for xml_name in tqdm(xml_names, desc="Processing"):
        try:
            with zf.open(xml_name) as f:
                raw = f.read()

            # Attempt 1: standard parser
            try:
                root = ET.fromstring(raw)
            except ET.ParseError:
                # Attempt 2: lxml recovery mode for malformed XML
                parser = LET.XMLParser(recover=True, encoding='utf-8')
                lxml_root = LET.fromstring(raw, parser=parser)
                root = ET.fromstring(LET.tostring(lxml_root))

            df = parse_xml_to_df(root)
            df = df.rename(columns=column_mapping)

            # Provenance
            df['source_file'] = xml_name
            df['source_url']  = 'https://udi.nmpa.gov.cn/download.html'

            # Date standardization (ISO 8601)
            for col in date_columns:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')

            # Write to SQLite (no dedup yet — handled globally after)
            df.to_sql('devices', conn, if_exists='append', index=False,
                      method='multi', chunksize=500)
            total_written += len(df)

        except Exception as e:
            error_files.append((xml_name, str(e)))
            print(f"\n⚠️  Skipped: {xml_name} — {e}")

conn.close()
print(f"\n✅ Initial load complete. Total rows written: {total_written:,}")
if error_files:
    print(f"⚠️  Files with errors: {len(error_files)}")

Processing:  42%|████████████████████████████▏                                      | 478/1136 [15:53<21:25,  1.95s/it]C:\Users\Ava\AppData\Local\Temp\ipykernel_30936\1361893028.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')
Processing:  70%|██████████████████████████████████████████████▋                    | 791/1136 [26:29<12:10,  2.12s/it]C:\Users\Ava\AppData\Local\Temp\ipykernel_30936\1361893028.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')
Processing:  83%|███████████████████████████████████████████████████████▊           | 947/1136 [31:50<06:22,


✅ Initial load complete. Total rows written: 5,674,158


### 4： Deduplication (Cross-file Duplicate Removal)

In [8]:
# Global deduplication across all files
# Rule: same device_record_key = exact same record loaded from multiple XML shards
# Keep first occurrence, remove all subsequent duplicates

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA journal_mode=WAL")

before = conn.execute("SELECT COUNT(*) FROM devices").fetchone()[0]
print(f"Rows before deduplication: {before:,}")

# Step 1: Add duplicate_flag column
conn.execute("ALTER TABLE devices ADD COLUMN duplicate_flag INTEGER DEFAULT 0")
conn.commit()

# Step 2: Flag duplicate rows (same device_record_key, keep first rowid)
conn.execute("""
    UPDATE devices
    SET duplicate_flag = 1
    WHERE rowid NOT IN (
        SELECT MIN(rowid)
        FROM devices
        GROUP BY device_record_key
    )
    AND device_record_key IS NOT NULL
""")
conn.commit()

flagged = conn.execute("SELECT COUNT(*) FROM devices WHERE duplicate_flag = 1").fetchone()[0]
print(f"Duplicate rows flagged: {flagged:,}")

# Step 3: Remove flagged duplicates
conn.execute("DELETE FROM devices WHERE duplicate_flag = 1")
conn.commit()

after = conn.execute("SELECT COUNT(*) FROM devices").fetchone()[0]
print(f"Rows after deduplication: {after:,}")
print(f"Rows removed: {before - after:,}")

conn.close()

Rows before deduplication: 5,674,158
Duplicate rows flagged: 445
Rows after deduplication: 5,673,713
Rows removed: 445


### 5： Validation

In [27]:
conn = sqlite3.connect(DB_PATH)

total = conn.execute("SELECT COUNT(*) FROM devices").fetchone()[0]
remaining_dups = conn.execute(
    "SELECT COUNT(*) FROM devices WHERE duplicate_flag = 1"
).fetchone()[0]
db_size = os.path.getsize(DB_PATH)

print(f"Final row count:        {total:,}")
print(f"Remaining duplicates:   {remaining_dups:,}")
print(f"Database size:          {db_size / 1024**3:.2f} GB")
print(f"Encoding:               UTF-8")
print(f"Date format:            ISO 8601 (YYYY-MM-DD)")

# Preview
pd.read_sql("""
    SELECT device_record_key, product_generic_name,
           registrant_name_cn, registration_or_filing_number,
           version_status, duplicate_flag
    FROM devices LIMIT 5
""", conn)

conn.close()

Final row count:        5,673,713
Remaining duplicates:   0
Database size:          3.99 GB
Encoding:               UTF-8
Date format:            ISO 8601 (YYYY-MM-DD)
